In [ ]:
import numpy as np

# ---------------------------------------------------------------------------
# 1. Synthesize Minimal Triage Clinical Data
# ---------------------------------------------------------------------------
np.random.seed(42)
N_patients = 250

# Continuous clinical triage vitals
hr = np.random.normal(90, 20, N_patients)     # Heart Rate
sbp = np.random.normal(120, 25, N_patients)   # Systolic BP
gcs = np.random.uniform(3, 15, N_patients)    # Glasgow Coma Scale

# Standardize inputs (crucial for stable HMC gradients and step sizing)
def standardize(v):
    return (v - np.mean(v)) / np.std(v)

X = np.column_stack([
    np.ones(N_patients),       # Intercept: beta_0
    standardize(hr),           # beta_1: higher HR -> higher risk
    standardize(sbp),          # beta_2: lower SBP -> shock risk
    standardize(gcs)           # beta_3: lower GCS -> neurological risk
])

# Generate synthetic binary 30-hour mortality outcomes
true_beta = np.array([-2.2, 0.8, -0.6, -1.1])
logits = X @ true_beta
prob = 1.0 / (1.0 + np.exp(-logits))
y = (np.random.rand(N_patients) < prob).astype(float)


# ---------------------------------------------------------------------------
# 2. Log-Posterior and Gradient (Equation 7 from the paper)
# ---------------------------------------------------------------------------
sigma_sq = 4.0  # Prior variance: beta ~ N(0, sigma_sq * I)

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))

def log_posterior(beta, X, y, sigma_sq):
    pi = sigmoid(X @ beta)
    # Log-likelihood (Bernoulli) + Log-prior (Normal)
    eps = 1e-12
    log_lik = np.sum(y * np.log(pi + eps) + (1.0 - y) * np.log(1.0 - pi + eps))
    log_prior = -0.5 * np.sum(beta**2) / sigma_sq
    return log_lik + log_prior

def grad_log_posterior(beta, X, y, sigma_sq):
    pi = sigmoid(X @ beta)
    # Grad = X^T * (y - pi) - beta / sigma_sq
    return X.T @ (y - pi) - (beta / sigma_sq)


# ---------------------------------------------------------------------------
# 3. Leapfrog Integrator (Numerical simulation of Hamilton's Equations)
# ---------------------------------------------------------------------------
def leapfrog(beta, p, epsilon, L, X, y, sigma_sq):
    beta_tilde = np.copy(beta)
    p_tilde = np.copy(p)

    # Half step for momentum
    p_tilde += 0.5 * epsilon * grad_log_posterior(beta_tilde, X, y, sigma_sq)

    # Alternate full steps
    for step in range(L):
        # Full step for position
        beta_tilde += epsilon * p_tilde
        # Full step for momentum (except final half-step)
        if step != L - 1:
            p_tilde += epsilon * grad_log_posterior(beta_tilde, X, y, sigma_sq)

    # Final half step for momentum
    p_tilde += 0.5 * epsilon * grad_log_posterior(beta_tilde, X, y, sigma_sq)
    return beta_tilde, -p_tilde  # Negate momentum for reversibility


# ---------------------------------------------------------------------------
# 4. Hamiltonian Monte Carlo Sampling Loop
# ---------------------------------------------------------------------------
def sample_hmc(X, y, n_samples=1500, epsilon=0.03, L=15, sigma_sq=4.0):
    k = X.shape[1]
    samples = np.zeros((n_samples, k))
    current_beta = np.zeros(k)  # Initial state
    accepted = 0

    for t in range(n_samples):
        # Step 1: Draw random momentum p ~ N(0, I)
        current_p = np.random.normal(0, 1, size=k)

        # Step 2: Leapfrog trajectory
        prop_beta, prop_p = leapfrog(current_beta, current_p, epsilon, L, X, y, sigma_sq)

        # Step 3: Compute Hamiltonian energies: H(beta, p) = -log_f(beta) + 0.5 * p^T * p
        current_U = -log_posterior(current_beta, X, y, sigma_sq)
        current_K = 0.5 * np.sum(current_p**2)

        prop_U = -log_posterior(prop_beta, X, y, sigma_sq)
        prop_K = 0.5 * np.sum(prop_p**2)

        # Step 4: Metropolis acceptance ratio
        log_alpha = (current_U + current_K) - (prop_U + prop_K)
        if np.log(np.random.uniform(0, 1)) < log_alpha:
            current_beta = prop_beta
            accepted += 1

        samples[t] = current_beta

    print(f"Acceptance rate: {accepted / n_samples:.2%}")
    return samples


# ---------------------------------------------------------------------------
# 5. Fit Model & Predict 30-Hour Mortality Risk
# ---------------------------------------------------------------------------
# Draw samples and discard the first 500 as warmup/burn-in
posterior_samples = sample_hmc(X, y, n_samples=1500, epsilon=0.03, L=15)[500:]

# Prediction for a new critical triage patient:
# Elevated HR (+1.8 std), Dropping SBP (-1.5 std), Low GCS (-2.0 std)
new_patient = np.array([1.0, 1.8, -1.5, -2.0])

# Posterior predictive probability distribution
sample_preds = sigmoid(posterior_samples @ new_patient)
mean_risk = np.mean(sample_preds)
ci_lower, ci_upper = np.percentile(sample_preds, [2.5, 97.5])

print(f"\nEstimated 30-hour mortality risk: {mean_risk:.1%}")
print(f"95% Credible Interval: [{ci_lower:.1%}, {ci_upper:.1%}]")